In [ ]:
!pip install datasets --quiet

from google.colab import files
print("Upload student_main_T4_lam0.9_best.pth (and optionally student_ablation_T2_lam0.9_best.pth)")
uploaded = files.upload()

Upload student_main_T4_lam0.9_best.pth (and optionally student_ablation_T2_lam0.9_best.pth)


Saving student_main_T4_lam0.9_best.pth to student_main_T4_lam0.9_best.pth


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from datasets import load_dataset
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
class TernarizeSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, W, clip_value):
        delta = 0.7 * W.abs().mean()
        pos_mask = W > delta
        neg_mask = W < -delta
        survivors = W[pos_mask | neg_mask]
        alpha = survivors.abs().mean() if survivors.numel() > 0 else torch.tensor(1e-8, device=W.device, dtype=W.dtype)
        W_ternary = torch.zeros_like(W)
        W_ternary[pos_mask] = alpha
        W_ternary[neg_mask] = -alpha
        ctx.save_for_backward(W)
        ctx.clip_value = clip_value
        return W_ternary

    @staticmethod
    def backward(ctx, grad_output):
        (W,) = ctx.saved_tensors
        pass_mask = (W.abs() <= ctx.clip_value).to(grad_output.dtype)
        return grad_output * pass_mask, None

def ternarize(W, clip_value=1.0):
    return TernarizeSTE.apply(W, clip_value)

def compute_twn_stats(W):
    with torch.no_grad():
        delta = 0.7 * W.abs().mean()
        pos_mask = W > delta
        neg_mask = W < -delta
        survivors = W[pos_mask | neg_mask]
        alpha = survivors.abs().mean().item() if survivors.numel() > 0 else 0.0
        sparsity = 1.0 - survivors.numel() / W.numel()
    return delta.item(), alpha, sparsity


class TernaryConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, bias=False, clip_value=1.0):
        super().__init__()
        self.stride = stride
        self.padding = padding
        self.clip_value = clip_value
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, kernel_size, kernel_size))
        nn.init.kaiming_uniform_(self.weight, a=5 ** 0.5)
        self.bias = nn.Parameter(torch.zeros(out_channels)) if bias else None

    def forward(self, x):
        W_ternary = ternarize(self.weight, self.clip_value)
        return F.conv2d(x, W_ternary, self.bias, stride=self.stride, padding=self.padding)

    def get_stats(self):
        return compute_twn_stats(self.weight)


class TernaryLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True, clip_value=1.0):
        super().__init__()
        self.clip_value = clip_value
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.weight, a=5 ** 0.5)
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

    def forward(self, x):
        W_ternary = ternarize(self.weight, self.clip_value)
        return F.linear(x, W_ternary, self.bias)

    def get_stats(self):
        return compute_twn_stats(self.weight)

In [ ]:
class TernaryBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_channels, out_channels, stride=1, clip_value=1.0):
        super().__init__()
        self.conv1 = TernaryConv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False, clip_value=clip_value)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = TernaryConv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False, clip_value=clip_value)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                TernaryConv2d(in_channels, out_channels, 1, stride=stride, bias=False, clip_value=clip_value),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class TernaryResNetCIFAR(nn.Module):
    def __init__(self, num_blocks, num_classes=10, clip_value=1.0):
        super().__init__()
        self.in_channels = 64
        self.clip_value = clip_value
        self.stem = nn.Sequential(
            TernaryConv2d(3, 64, 3, stride=1, padding=1, bias=False, clip_value=clip_value),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True)
        )
        self.stage1 = self._make_stage(64, num_blocks[0], stride=1)
        self.stage2 = self._make_stage(128, num_blocks[1], stride=2)
        self.stage3 = self._make_stage(256, num_blocks[2], stride=2)
        self.stage4 = self._make_stage(512, num_blocks[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = TernaryLinear(512, num_classes, bias=True, clip_value=clip_value)

    def _make_stage(self, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(TernaryBasicBlock(self.in_channels, out_channels, stride=s, clip_value=self.clip_value))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.stem(x)
        out = self.stage1(out); out = self.stage2(out); out = self.stage3(out); out = self.stage4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        return self.fc(out)

def ResNet18Ternary(num_classes=10, clip_value=1.0):
    return TernaryResNetCIFAR(num_blocks=[2, 2, 2, 2], num_classes=num_classes, clip_value=clip_value)

In [ ]:
CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

ds = load_dataset("EleutherAI/cifarnet")
cifarnet_test = ds["test"]

per_class_images = defaultdict(list)
target_per_class = 10
for example in cifarnet_test:
    label = example["label"]
    if len(per_class_images[label]) < target_per_class:
        per_class_images[label].append(example["img"])
    if len(per_class_images) == 10 and all(len(v) >= target_per_class for v in per_class_images.values()):
        break

for i in range(10):
    print(f"{CIFAR10_CLASSES[i]}: {len(per_class_images[i])} images")

README.md:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

data/train-00000-of-00004-479ecf2b550e1a(…): reconstructing file:   0%|          |  0.00B /  390MB            

data/train-00000-of-00004-479ecf2b550e1a(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-3298bc3dea6547(…): reconstructing file:   0%|          |  0.00B /  390MB            

data/train-00001-of-00004-3298bc3dea6547(…): downloading bytes:           |  0.00B            

...

data/test-00000-of-00001-6109c632108adc2(…): reconstructing file:   0%|          |  0.00B / 82.2MB            

data/test-00000-of-00001-6109c632108adc2(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/190000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

airplane: 10 images
automobile: 10 images
bird: 10 images
cat: 10 images
deer: 10 images
dog: 10 images
frog: 10 images
horse: 10 images
ship: 10 images
truck: 10 images


In [ ]:
cifarnet_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

all_images, all_true_labels = [], []
for cls_idx in range(10):
    for img in per_class_images[cls_idx]:
        all_images.append(cifarnet_transform(img.convert("RGB")))
        all_true_labels.append(cls_idx)

cifarnet_batch = torch.stack(all_images).to(device)
cifarnet_labels = torch.tensor(all_true_labels).to(device)
print(f"Batch shape: {cifarnet_batch.shape} (expect [100, 3, 32, 32])")

Batch shape: torch.Size([100, 3, 32, 32]) (expect [100, 3, 32, 32])


In [ ]:
def evaluate_checkpoint_on_cifarnet(ckpt_path, label):
    model = ResNet18Ternary(num_classes=10, clip_value=1.0).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    with torch.no_grad():
        logits = model(cifarnet_batch)
        preds = logits.argmax(dim=1)

    overall_acc = (preds == cifarnet_labels).float().mean().item() * 100
    print(f"\n[{label}] Overall CIFARNet accuracy: {overall_acc:.2f}%")
    per_class = {}
    for cls_idx in range(10):
        mask = cifarnet_labels == cls_idx
        cls_acc = (preds[mask] == cifarnet_labels[mask]).float().mean().item() * 100
        per_class[CIFAR10_CLASSES[cls_idx]] = cls_acc
        print(f"  {CIFAR10_CLASSES[cls_idx]:<12} {cls_acc:.1f}%")

    return {"label": label, "overall_acc": overall_acc, "per_class_acc": per_class,
            "preds": preds.cpu().tolist(), "labels": cifarnet_labels.cpu().tolist()}

# Primary result (this is the one for your main report)
main_result = evaluate_checkpoint_on_cifarnet("student_main_T4_lam0.9_best.pth", "Main (T=4)")


[Main (T=4)] Overall CIFARNet accuracy: 78.00%
  airplane     80.0%
  automobile   80.0%
  bird         90.0%
  cat          60.0%
  deer         80.0%
  dog          50.0%
  frog         90.0%
  horse        70.0%
  ship         90.0%
  truck        90.0%


In [ ]:
with open("cifarnet_ood_results.json", "w") as f:
    json.dump({"main": main_result, "ablation": ablation_result if 'ablation_result' in dir() else None}, f, indent=2)

from google.colab import files
files.download("cifarnet_ood_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>